# Week 8 — Retrieval Evaluation
> Goal: Measure retrieval and generation quality using classical IR metrics and RAGAS, then iterate to improve scores.

**Topics covered:**
- Classical IR: Precision@k, Recall@k, MRR, NDCG
- RAGAS: Faithfulness, Answer Relevancy, Context Precision, Context Recall
- Chunking strategy A/B comparison
- Error analysis: what causes low scores?
- Improvement loop: diagnose → fix → re-evaluate


## 0. Setup

In [ ]:
import sys, os, json
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from evaluator import (
    precision_at_k, recall_at_k, mrr, ndcg_at_k,
    average_precision, mean_average_precision, evaluate_retrieval
)
from embedder import get_embedder
from vectorstore import FAISSStore
from chunker import chunk_document

print('Setup complete.')

---
## 1. Classical IR Metrics — Intuition

In [ ]:
# ── 1.1  Understand each metric with a concrete example ──────────────────────
# Imagine: there are 3 relevant documents in the corpus.
# Your retriever returned these 8 documents in order:
retrieved = ['doc_1', 'doc_4', 'doc_2', 'doc_7', 'doc_3', 'doc_8', 'doc_9', 'doc_5']
relevant  = {'doc_1', 'doc_2', 'doc_3'}   # the 3 truly relevant docs

print('Retrieved:', retrieved)
print('Relevant: ', relevant)
print()

for k in [1, 3, 5, 8]:
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    print(f'k={k}:  Precision@{k}={p:.3f}  Recall@{k}={r:.3f}')

ap = average_precision(retrieved, relevant)
print(f'\nAverage Precision (AP): {ap:.4f}')
print('  AP rewards finding relevant docs at higher ranks.')

In [ ]:
# ── 1.2  MRR — Mean Reciprocal Rank ──────────────────────────────────────────
# MRR = average of 1/rank_of_first_relevant_doc across queries

multi_query_results = [
    # (retrieved_list, relevant_set)     ← first relevant at rank...
    (['doc_1', 'doc_4', 'doc_2'],  {'doc_1', 'doc_2'}),   # rank 1  → RR = 1.000
    (['doc_4', 'doc_7', 'doc_3'],  {'doc_3'}),             # rank 3  → RR = 0.333
    (['doc_8', 'doc_9', 'doc_5'],  {'doc_1'}),             # not found → RR = 0.000
    (['doc_6', 'doc_1', 'doc_2'],  {'doc_1', 'doc_2'}),   # rank 2  → RR = 0.500
]

mrr_score = mrr(multi_query_results)
print(f'MRR = {mrr_score:.4f}')
print(f'MAP = {mean_average_precision(multi_query_results):.4f}')
print()
print('RR per query:')
for i, (ret, rel) in enumerate(multi_query_results):
    first_hit = next((1/(r+1) for r, d in enumerate(ret) if d in rel), 0)
    print(f'  Q{i+1}: retrieved={ret}  → first hit rank={int(1/first_hit) if first_hit else "none"}, RR={first_hit:.3f}')

In [ ]:
# ── 1.3  NDCG — handles graded relevance ─────────────────────────────────────
# Some documents are MORE relevant than others.
retrieved_ids = ['doc_A', 'doc_B', 'doc_C', 'doc_D', 'doc_E']
relevance_scores = {
    'doc_A': 3.0,   # highly relevant (answers the question precisely)
    'doc_C': 2.0,   # relevant (related section)
    'doc_E': 1.0,   # somewhat relevant (tangential)
    # doc_B, doc_D not relevant (score=0)
}

ndcg_3 = ndcg_at_k(retrieved_ids, relevance_scores, k=3)
ndcg_5 = ndcg_at_k(retrieved_ids, relevance_scores, k=5)
print(f'NDCG@3 = {ndcg_3:.4f}')
print(f'NDCG@5 = {ndcg_5:.4f}')

# If we reorder to put highly relevant docs first:
perfect_order = ['doc_A', 'doc_C', 'doc_E', 'doc_B', 'doc_D']
ndcg_perfect = ndcg_at_k(perfect_order, relevance_scores, k=5)
print(f'NDCG@5 (perfect order) = {ndcg_perfect:.4f}  ← this is the ceiling')

---
## 2. Evaluate Our Financial Retriever

In [ ]:
# ── 2.1  Build a test corpus and evaluation set ───────────────────────────────
CORPUS = [
    {'id': 'aapl_rev_2023',    'text': "Apple total net sales for fiscal 2023 were $383.3 billion, a 3% decrease.", 'company': 'AAPL', 'year': 2023, 'section': 'Financial Statements'},
    {'id': 'aapl_gm_2023',     'text': "Apple gross margin was 44.1% in fiscal 2023, compared to 43.3% in 2022.",  'company': 'AAPL', 'year': 2023, 'section': 'MDA'},
    {'id': 'aapl_svc_2023',    'text': "Services net sales were $85.2 billion, growing 9% year over year.",         'company': 'AAPL', 'year': 2023, 'section': 'MDA'},
    {'id': 'aapl_risk_comp',   'text': "Apple faces intense competition from Samsung, Google, and Microsoft.",       'company': 'AAPL', 'year': 2023, 'section': 'Risk Factors'},
    {'id': 'aapl_risk_econ',   'text': "Global economic conditions could adversely affect Apple's business.",       'company': 'AAPL', 'year': 2023, 'section': 'Risk Factors'},
    {'id': 'msft_rev_2023',    'text': "Microsoft total revenue was $211.9 billion for fiscal 2023, up 7%.",         'company': 'MSFT', 'year': 2023, 'section': 'Financial Statements'},
    {'id': 'msft_cloud_2023',  'text': "Microsoft Intelligent Cloud segment revenue was $87.9B, growing 19%.",      'company': 'MSFT', 'year': 2023, 'section': 'Business'},
    {'id': 'msft_azure_2023',  'text': "Azure and other cloud services revenue grew 29 percent.",                    'company': 'MSFT', 'year': 2023, 'section': 'Business'},
    {'id': 'msft_gm_2023',     'text': "Microsoft gross margin increased to 69% in fiscal 2023, up from 68%.",      'company': 'MSFT', 'year': 2023, 'section': 'MDA'},
    {'id': 'nvda_dc_2024',     'text': "Nvidia data center revenue for fiscal 2024 was $47.5 billion.",              'company': 'NVDA', 'year': 2024, 'section': 'Business'},
    {'id': 'nvda_risk_comp',   'text': "Nvidia faces intense competition from AMD and Intel.",                       'company': 'NVDA', 'year': 2024, 'section': 'Risk Factors'},
    {'id': 'aapl_iphone_2023', 'text': "iPhone net sales were $200.6 billion in fiscal year 2023.",                  'company': 'AAPL', 'year': 2023, 'section': 'Financial Statements'},
]

EVAL_SET = [
    {'query': 'Apple total revenue 2023',                          'relevant_ids': ['aapl_rev_2023', 'aapl_iphone_2023']},
    {'query': 'What was Apple gross margin?',                      'relevant_ids': ['aapl_gm_2023']},
    {'query': 'Apple Services revenue growth',                     'relevant_ids': ['aapl_svc_2023']},
    {'query': 'What competition risks does Apple face?',           'relevant_ids': ['aapl_risk_comp', 'aapl_risk_econ']},
    {'query': 'Microsoft cloud revenue',                           'relevant_ids': ['msft_cloud_2023', 'msft_azure_2023', 'msft_rev_2023']},
    {'query': 'Azure growth rate',                                 'relevant_ids': ['msft_azure_2023', 'msft_cloud_2023']},
    {'query': 'Compare Apple and Microsoft gross margins',         'relevant_ids': ['aapl_gm_2023', 'msft_gm_2023']},
    {'query': 'Nvidia data center performance',                    'relevant_ids': ['nvda_dc_2024']},
    {'query': 'Apple iPhone revenue',                              'relevant_ids': ['aapl_iphone_2023', 'aapl_rev_2023']},
    {'query': 'Who are Nvidia main competitors?',                  'relevant_ids': ['nvda_risk_comp']},
]

# Embed corpus
embedder = get_embedder('openai-small')
corpus_texts = [c['text'] for c in CORPUS]
corpus_vecs  = embedder(corpus_texts)

store = FAISSStore(dim=corpus_vecs.shape[1])
store.add(corpus_vecs, CORPUS)

print(f'Corpus: {len(CORPUS)} docs | Eval set: {len(EVAL_SET)} queries')

In [ ]:
# ── 2.2  Run evaluation ───────────────────────────────────────────────────────
def retriever_fn(query: str, k: int = 5) -> list[str]:
    q_vec = embedder([query])[0]
    results = store.search(q_vec, k=k)
    return [r['id'] for r in results]

metrics = evaluate_retrieval(retriever_fn, EVAL_SET, k_values=[1, 3, 5])

print('\n=== Retrieval Evaluation Results ===')
for key, val in metrics.items():
    if isinstance(val, dict):
        for metric, score in val.items():
            bar = '█' * int(score * 20)
            print(f'  {metric:<22} {score:.4f}  {bar}')
    else:
        bar = '█' * int(val * 20)
        print(f'  {key:<22} {val:.4f}  {bar}')

In [ ]:
# ── 2.3  Per-query breakdown ──────────────────────────────────────────────────
os.makedirs('plots', exist_ok=True)

per_query_p5  = []
per_query_r5  = []
per_query_mrr = []

for qitem in EVAL_SET:
    retrieved = retriever_fn(qitem['query'], k=5)
    relevant  = set(qitem['relevant_ids'])
    per_query_p5.append(precision_at_k(retrieved, relevant, 5))
    per_query_r5.append(recall_at_k(retrieved, relevant, 5))
    rr = 1 / (retrieved.index(next((d for d in retrieved if d in relevant), None)) + 1) \
         if any(d in relevant for d in retrieved) else 0
    per_query_mrr.append(rr)

query_labels = [q['query'][:30] + '…' for q in EVAL_SET]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(EVAL_SET))
w = 0.28
ax.bar(x - w, per_query_p5,  width=w, label='Precision@5', color='#2196F3', alpha=0.85)
ax.bar(x,     per_query_r5,  width=w, label='Recall@5',    color='#4CAF50', alpha=0.85)
ax.bar(x + w, per_query_mrr, width=w, label='RR',          color='#FF9800', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(query_labels, rotation=40, ha='right', fontsize=8)
ax.set_ylim(0, 1.15)
ax.axhline(np.mean(per_query_p5), color='#2196F3', linestyle='--', alpha=0.5)
ax.axhline(np.mean(per_query_r5), color='#4CAF50', linestyle='--', alpha=0.5)
ax.set_ylabel('Score')
ax.set_title('Per-Query Retrieval Metrics', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/per_query_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. RAGAS — LLM-as-Judge Evaluation

In [ ]:
# ── 3.1  Load the chain and run RAGAS ─────────────────────────────────────────
# Make sure the index is built first:
#   python build_index.py --demo

INDEX_PATH = '../data/indices/financial_index'

if not os.path.exists(f'{INDEX_PATH}.faiss'):
    print('Index not found — building demo index...')
    os.system('cd .. && python build_index.py --demo')

In [ ]:
# ── 3.2  Manually run RAGAS on a few questions ────────────────────────────────
# This cell shows the raw RAGAS mechanics without needing the full chain.

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# A small manually-curated evaluation set
eval_rows = [
    {
        'question':    'What was Apple total revenue in fiscal 2023?',
        'answer':      'Apple total net sales in fiscal 2023 were $383.3 billion, a decrease of 3% from $394.3 billion in 2022.',
        'contexts':    ['Apple total net sales for fiscal 2023 were $383.3 billion, a 3% decrease.',
                        'iPhone net sales were $200.6 billion in fiscal year 2023.'],
        'ground_truth': 'Apple total net sales in fiscal 2023 were $383.3 billion.',
    },
    {
        'question':    'What drove Apple gross margin improvement?',
        'answer':      'Apple gross margin improved due to a favorable mix shift toward higher-margin Services revenue. Services gross margin was 70.8% while Products was 36.6%.',
        'contexts':    ['Apple gross margin was 44.1% in fiscal 2023, compared to 43.3% in 2022.',
                        'Services gross margin was 70.8% while Products gross margin was 36.6%.'],
        'ground_truth': 'Apple gross margin improved from 43.3% to 44.1% driven by growth in higher-margin Services revenue.',
    },
    {
        'question':    'How did Microsoft Azure perform?',
        'answer':      'Azure and other cloud services revenue grew 29 percent in fiscal 2023.',
        'contexts':    ['Microsoft Intelligent Cloud segment revenue was $87.9B, growing 19%.',
                        'Azure and other cloud services revenue grew 29 percent.'],
        'ground_truth': 'Azure revenue grew 29 percent in fiscal year 2023.',
    },
    {
        'question':    'What risks did Apple highlight related to supply chain?',
        'answer':      'Apple depends on third-party manufacturers. If supply or quality of components is constrained, revenue and gross margins could be adversely affected.',
        'contexts':    ['Apple faces intense competition from Samsung, Google, and Microsoft.',
                        'Global economic conditions could adversely affect Apple business.'],  # ← wrong context
        'ground_truth': 'Apple disclosed dependency on third-party manufacturers as a key supply chain risk.',
    },
]

ragas_dataset = Dataset.from_list(eval_rows)

result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

df = result.to_pandas()
print('RAGAS results per question:')
print(df[['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']].to_string(index=False))

In [ ]:
# ── 3.3  Visualize RAGAS scores ────────────────────────────────────────────────
metrics_cols = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
means = df[metrics_cols].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Mean scores bar chart
target_scores = [0.90, 0.85, 0.80, 0.60]
bar_colors = ['#4CAF50' if v >= t else '#FF5722' for v, t in zip(means.values, target_scores)]
bars = ax1.bar(metrics_cols, means.values, color=bar_colors, alpha=0.85, edgecolor='white')
for t, x in zip(target_scores, range(len(metrics_cols))):
    ax1.axhline(y=t, xmin=x/len(metrics_cols)+0.01, xmax=(x+1)/len(metrics_cols)-0.01,
                color='black', linewidth=2, linestyle='--', alpha=0.6)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('Mean Score')
ax1.set_title('RAGAS Mean Scores\n(dashed line = target threshold)')
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, means.values):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center', fontsize=10)

# Heatmap: questions × metrics
data = df[metrics_cols].values
q_labels_short = [f'Q{i+1}: {r["question"][:25]}…' for i, r in enumerate(eval_rows)]
im = ax2.imshow(data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(len(metrics_cols)))
ax2.set_xticklabels([m.replace('_', ' ').title() for m in metrics_cols], rotation=25, ha='right', fontsize=9)
ax2.set_yticks(range(len(eval_rows)))
ax2.set_yticklabels(q_labels_short, fontsize=8)
ax2.set_title('RAGAS Scores Per Question')
plt.colorbar(im, ax=ax2, label='Score')
for i in range(len(eval_rows)):
    for j in range(len(metrics_cols)):
        ax2.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center', fontsize=9)

plt.suptitle('RAGAS Evaluation Results', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plots/ragas_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Error Analysis

In [ ]:
# ── 4.1  Identify the worst-performing questions ─────────────────────────────
df['avg_score'] = df[metrics_cols].mean(axis=1)
df['question_short'] = [q['question'] for q in eval_rows]

sorted_df = df.sort_values('avg_score')
print('Worst-performing questions (lowest average RAGAS score):')
print(sorted_df[['question_short', 'faithfulness', 'context_precision', 'avg_score']].to_string(index=False))

print('\n=== Diagnosis for Q4 (supply chain risk, wrong context retrieved) ===')
print('Problem: The retriever returned competition/economic context instead of supply chain context.')
print('Root cause: "supply chain" keywords not in corpus OR chunking split that section away.')
print('Fix options:')
print('  1. Add more supply-chain text to the index')
print('  2. Use semantic chunking to keep supply-chain discussion intact')
print('  3. Add keyword (BM25) fallback for specific term queries')

---
## 5. Improvement Loop: Compare Strategies

In [ ]:
# ── 5.1  A/B test: recursive vs sentence-window chunking ─────────────────────
LONG_TEXT = "\n\n".join(c['text'] for c in CORPUS)  # combine all corpus texts

results_ab = {}
for strategy in ['recursive', 'sentence_window']:
    chunks = chunk_document(LONG_TEXT, strategy=strategy, metadata={'source': 'combined'})
    texts  = [c.text for c in chunks]
    chunk_vecs = embedder(texts)
    
    ab_store = FAISSStore(dim=chunk_vecs.shape[1])
    chunk_meta = [{'id': f'{strategy}_{i}', 'text': t} for i, t in enumerate(texts)]
    ab_store.add(chunk_vecs, chunk_meta)
    
    # Since we don't have doc-level IDs per chunk, use keyword overlap as proxy
    hit_rates = []
    for qitem in EVAL_SET:
        q_vec = embedder([qitem['query']])[0]
        top5 = ab_store.search(q_vec, k=5)
        combined = ' '.join(r['text'].lower() for r in top5)
        # Use first relevant doc text as keyword probe
        probe_id = qitem['relevant_ids'][0]
        probe_text = next(c['text'] for c in CORPUS if c['id'] == probe_id)
        key_words = [w for w in probe_text.lower().split() if len(w) > 5][:4]
        hits = sum(1 for kw in key_words if kw in combined)
        hit_rates.append(hits / max(len(key_words), 1))
    
    avg_hit = np.mean(hit_rates)
    results_ab[strategy] = {'hit_rate': avg_hit, 'n_chunks': len(chunks)}
    print(f'[{strategy:20s}] keyword hit rate={avg_hit:.3f}  chunks={len(chunks)}')

winner = max(results_ab, key=lambda x: results_ab[x]['hit_rate'])
print(f'\nBetter strategy for this corpus: {winner}')

In [ ]:
# ── 5.2  Visualize: chunking strategies over k values ─────────────────────────
k_values = [1, 3, 5, 10]

# Use our main store with document-level IDs
precision_by_k = []
recall_by_k    = []

for k in k_values:
    p_scores, r_scores = [], []
    for qitem in EVAL_SET:
        retrieved_ids = retriever_fn(qitem['query'], k=k)
        relevant = set(qitem['relevant_ids'])
        p_scores.append(precision_at_k(retrieved_ids, relevant, k))
        r_scores.append(recall_at_k(retrieved_ids, relevant, k))
    precision_by_k.append(np.mean(p_scores))
    recall_by_k.append(np.mean(r_scores))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, precision_by_k, 'o-', color='#2196F3', linewidth=2, label='Precision@k', markersize=8)
ax.plot(k_values, recall_by_k,    's-', color='#4CAF50', linewidth=2, label='Recall@k',    markersize=8)
for k, p, r in zip(k_values, precision_by_k, recall_by_k):
    ax.annotate(f'{p:.2f}', (k, p), textcoords='offset points', xytext=(0, 10), fontsize=9, color='#2196F3')
    ax.annotate(f'{r:.2f}', (k, r), textcoords='offset points', xytext=(0,-15), fontsize=9, color='#4CAF50')
ax.set_xlabel('k (number of retrieved chunks)')
ax.set_ylabel('Score')
ax.set_title('Precision@k vs Recall@k Trade-off', fontsize=12)
ax.set_xticks(k_values)
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('plots/precision_recall_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: As k increases, recall goes up but precision goes down.')
print('Choose k=5 as a good balance for financial Q&A.')

---
## 6. Final Report

In [ ]:
# ── 6.1  Print a clean summary ────────────────────────────────────────────────
print('=' * 60)
print('FINANCIAL RAG — WEEK 8 EVALUATION SUMMARY')
print('=' * 60)

print('\n📊 Classical Retrieval Metrics (k=5):')
for key, val in metrics.items():
    if isinstance(val, dict):
        for m, s in val.items():
            status = '✓' if s >= 0.6 else '✗'
            print(f'  {status} {m:<22} {s:.4f}')
    else:
        status = '✓' if val >= 0.7 else '✗'
        print(f'  {status} {key:<22} {val:.4f}')

print('\n🤖 RAGAS Metrics (LLM-as-judge):')
target_map = {'faithfulness': 0.90, 'answer_relevancy': 0.85,
              'context_precision': 0.80, 'context_recall': 0.60}
for m in metrics_cols:
    score = df[m].mean()
    target = target_map[m]
    status = '✓' if score >= target else '✗'
    print(f'  {status} {m:<22} {score:.4f}  (target: >{target})')

print('\n📌 Key Findings:')
print('  • Recursive chunking is a solid baseline')
print('  • Sentence-window chunking improves context_recall')
print('  • Low faithfulness = hallucination → use stricter prompt')
print('  • Low context_precision = noisy retrieval → tune MMR lambda')
print('\n🚀 Next steps:')
print('  1. Add BM25 sparse retrieval + RRF fusion')
print('  2. Fine-tune embeddings on financial domain')
print('  3. Implement query rewriting for multi-hop questions')
print('  4. Deploy to HuggingFace Spaces: python src/app.py')